In [ ]:
import numpy as np
from  MPCModelFuncs import *
from bagpy import bagreader
import pandas as pd
import yaml
import os
import matplotlib.pyplot as plt

bag = '/home/david/bags/autox_track_comp/autocross_2023-08-10-14-02-11.bag'
h = 0.05

idx_first = 100
idxs = [idx_first, idx_first+51]

In [ ]:
b = bagreader(bag)    
NN_DATA_MSG = b.message_by_topic('/control/learning_mpcc/nn_data')
nn_data = pd.read_csv(NN_DATA_MSG)

throttles = np.array(nn_data['control_cmd.throttle'])
steerings = np.array(nn_data['control_cmd.steering_angle'])
prev_throttles = np.array(nn_data['prev_control_cmd.throttle'])
prev_steerings = np.array(nn_data['prev_control_cmd.steering_angle'])
vx = np.array(nn_data['car_velocity.velocity.x'])
vy_raw = np.array(nn_data['car_velocity.velocity.y'])
r = np.array(nn_data['car_velocity.velocity.theta'])
vx_prev = np.array(nn_data['prev_car_velocity.velocity.x'])
vy_prev = np.array(nn_data['prev_car_velocity.velocity.y'])
X = np.array(nn_data['car_pose.x'])
Y = np.array(nn_data['car_pose.y'])
psi = np.array(nn_data['car_pose.theta'])
ax_imu = np.array(nn_data['car_acceleration.x'])
ay_imu = np.array(nn_data['car_acceleration.y'])

IMU_DATA_MSG = b.message_by_topic('/estimation/accel_gravity')
imu_data = pd.read_csv(IMU_DATA_MSG)
raw_ax_imu = np.array(imu_data['linear_acceleration.x'])
raw_ay_imu = np.array(imu_data['linear_acceleration.y'])

#vy = vy_raw + r*(0.665+0.44)
vy = vy_raw

In [ ]:
# plt.plot(vy)
# plt.plot(vy_mod)

# plt.legend(['vy', 'vy_mod'])
plt.plot(vy_raw)
plt.plot(vy)
plt.legend(['vy_raw', 'vy'])

In [ ]:
# Read yaml file
home_directory = os.path.expanduser("~")
p = np.zeros(24)
with open(home_directory + '/fst/autonomous-systems/src/control/learning_mpcc/config/default.yaml') as stream:
    parameters = yaml.load(stream, Loader=yaml.SafeLoader)
    p[0] = parameters['model_params']['l_f']
    p[1] = parameters['model_params']['l_r']
    p[2] = parameters['model_params']['m']
    p[3] = parameters['model_params']['I_z']
    p[4] = parameters['model_params']['T_max_front']
    p[5] = parameters['model_params']['T_max_rear']
    p[6] = parameters['model_params']['T_brake_front']
    p[7] = parameters['model_params']['T_brake_rear']
    p[8] = parameters['model_params']['GR']
    p[9] = parameters['model_params']['eta_motor']
    p[10] = parameters['model_params']['r_wheel']
    p[11] = parameters['model_params']['g']
    p[12] = parameters['model_params']['C_roll']
    p[13] = parameters['model_params']['rho']
    p[14] = parameters['model_params']['C_d']
    p[15] = parameters['model_params']['C_l']
    p[16] = parameters['tyre_params']['B']
    p[17] = parameters['tyre_params']['C']
    p[18] = parameters['tyre_params']['D']
    p[19] = parameters['model_params']['downforce_front']
    p[20] = parameters['model_params']['downforce_rear']
    p[21] = parameters['model_params']['h_cog']
    p[22] = parameters['model_params']['track_width']
    p[23] = parameters['model_params']['diff_gain']
    vel_min = parameters['model_params']['lambda_blend_min']

In [ ]:
states = np.array([vx, vy, r, vx_prev, vy_prev]).T
input = np.array([throttles, steerings, prev_throttles, prev_steerings]).T

x_kin, x_dyn, x_planar, x_simple_dyn = [states[idxs[0],:]], [states[idxs[0],:]], [states[idxs[0],:]], [states[idxs[0],:]]
i = 0

for idx in range(idxs[0],idxs[1]):
    curr_input = np.expand_dims(input[idx,:], axis=0)
    curr_kin = np.expand_dims(x_kin[i], axis=0)
    curr_dyn = np.expand_dims(x_dyn[i], axis=0)
    curr_planar = np.expand_dims(x_planar[i], axis=0)
    curr_simp_dyn = np.expand_dims(x_simple_dyn[i], axis=0)
    
    # Kinematic
    x_kin_append = np.squeeze(kinematic_mpc_model(curr_kin, curr_input, p, h))
    x_kin.append(x_kin_append)
    # Dynamical Bicycle
    dX_dyn = differential_mpc_model_data_diff_and_load(curr_dyn, curr_input, p)
    x_dyn_append = np.squeeze(np.expand_dims(curr_dyn[0,0:3],axis=1) + h*dX_dyn)
    x_dyn.append(x_dyn_append)
    # Planar 
    dX_planar = planar_model_TV(curr_planar, curr_input, p, h)
    euler_planar = np.expand_dims(curr_planar[0,0:3],axis=1) + h*dX_planar
    x_planar.append(np.hstack((euler_planar[:,0],curr_planar[0,0:2])))
    # Simple Dynamic
    dX_simple_dyn = dynamical_differential(curr_simp_dyn, curr_input, p, h)
    euler_simple_dyn = np.expand_dims(curr_simp_dyn[0,0:3],axis=1) + h*dX_simple_dyn
    x_simple_dyn.append(np.hstack((euler_simple_dyn[:,0],curr_simp_dyn[0,0:2])))
    
    i += 1

In [ ]:
vx_kin, vx_dyn, vx_planar, vx_simple_dyn = np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_simple_dyn),1))
vy_kin, vy_dyn, vy_planar, vy_simple_dyn = np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_simple_dyn),1))
r_kin, r_dyn, r_planar, r_simple_dyn = np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_kin),1)), np.zeros((len(x_simple_dyn),1))

for i in range(len(x_kin)):
    vx_kin[i] = x_kin[i][0]
    vx_dyn[i] = x_dyn[i][0]
    vx_planar[i] = x_planar[i][0]
    vx_simple_dyn[i] = x_simple_dyn[i][0]
    
    vy_kin[i] = x_kin[i][1]
    vy_dyn[i] = x_dyn[i][1]
    vy_planar[i] = x_planar[i][1]
    vy_simple_dyn[i] = x_simple_dyn[i][1]
    
    r_kin[i] = x_kin[i][2]
    r_dyn[i] = x_dyn[i][2]
    r_planar[i] = x_planar[i][2]
    r_simple_dyn[i] = x_simple_dyn[i][2]

fig, (ax1, ax2, ax3) = plt.subplots(3,1,figsize=(10,10))

#plt.plot(vx_kin)
ax1.plot(vx_dyn,label = 'dyn')
ax1.plot(vx_planar,label = 'planar')
ax1.plot(vx_simple_dyn,label = 'simple dyn')
ax1.plot(states[idxs[0]:idxs[1]+1,0], label = 'true')

ax1.legend()

ax2.plot(vy_dyn,label = 'dyn')
ax2.plot(vy_planar,label = 'planar')
ax2.plot(vy_simple_dyn,label = 'simple dyn')
ax2.plot(states[idxs[0]:idxs[1]+1,1], label = 'true')

ax2.legend()

ax3.plot(r_dyn,label = 'dyn')
ax3.plot(r_planar,label = 'planar')
ax3.plot(r_simple_dyn,label = 'simple dyn')
ax3.plot(states[idxs[0]:idxs[1]+1,2], label = 'true')

ax3.legend()

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(10,10))

ax1.plot(input[idxs[0]:idxs[1]+1,0])
ax2.plot(input[idxs[0]:idxs[1]+1,1])

In [ ]:
plt.plot(states[idxs[0]:idxs[1]+1,0])

In [ ]:
# x = np.array([1, 0.1, 0.1, 0.8, 0.1, 0])
# u = np.array([0.1, 0.2])

x = np.array([5.58024, 0.00958922, 0.0119762, 5.44626, 0.0191936])
u = np.array([0.757641, -0.000766104])


x = np.expand_dims(x, axis=0)
u = np.expand_dims(u, axis=0)


planar_model =  planar_model_TV(x, u, p, h)

print(planar_model)